In [ ]:
from bs4 import BeautifulSoup as bs
import pandas as pd
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
options.page_load_strategy = 'eager'
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)

data = []
max_articles = 20
page = 1

while len(data) < max_articles:
    url_feed = f'https://habr.com/ru/feed/page{page}/'
    driver.get(url_feed)
    soup = bs(driver.page_source, 'html.parser')
    
    articles = soup.find_all('article', class_='tm-articles-list__item')
    
    for article in articles:
        if len(data) >= max_articles:
            break
            
        try:
            link_tag = article.find('a', class_='tm-title__link')
            if not link_tag:
                continue
                
            article_url = f"https://habr.com{link_tag['href']}"
            driver.get(article_url)
            article_soup = bs(driver.page_source, 'html.parser')
            
            company_tag = article_soup.find('a', class_='tm-company-card__name')
            if not company_tag:
                continue 
                
            title = article_soup.find('h1').get_text(strip=True)
            company_name = company_tag.get_text(strip=True)
            
            desc_tag = article_soup.find('div', class_='tm-company-card__description')
            description = desc_tag.get_text(strip=True) if desc_tag else "Отсутствует"
            
            rating = article_soup.find('div', class_='counter').get_text(strip=True)
            
            hubs = [hub.get_text(strip=True) for hub in article_soup.find_all('span', class_='tm-publication-hub__link-container')]
            fields = ", ".join(hubs)
            
            date_tag = article_soup.find('span', class_='tm-article-datetime-published').find('time')
            date_val = date_tag['title'].split(',')[0] if date_tag else "Не указана"
            
            body_tag = article_soup.find('div', class_='article-formatted-body')
            text_content = re.sub(r'\s+', ' ', body_tag.get_text()).strip() if body_tag else ""

            data.append({
                'title': title,
                'namecompany': company_name,
                'description': description,
                'rating': rating,
                'field': fields,
                'date': date_val,
                'textpub': text_content
            })
            
            print(f"{len(data)}. Успешно обработано: {article_url}")
            
        except Exception as e:
            print(f"Ошибка при обработке статьи: {e}")
            continue
            
    page += 1

driver.quit()
df = pd.DataFrame(data)
df.head()
df.to_csv('habr_data.csv', index=False)
print("\nГотово")

In [ ]:
df.info()